In [1]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/2025-1/Proyecto integrador /dataset-tickets-multi-lang-4-20k.csv")
df.info()
df.head()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 15 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   subject   18539 non-null  object
 1   body      19998 non-null  object
 2   answer    19996 non-null  object
 3   type      20000 non-null  object
 4   queue     20000 non-null  object
 5   priority  20000 non-null  object
 6   language  20000 non-null  object
 7   tag_1     20000 non-null  object
 8   tag_2     19954 non-null  object
 9   tag_3     19905 non-null  object
 10  tag_4     18461 non-null  object
 11  tag_5     13091 non-null  object
 12  tag_6     7351 non-null   object
 13  tag_7     3928 non-null   object
 14  tag_8     1907 non-null   object
dtypes: object(15)
memory usage: 2.3+ MB


,subject,body,answer,type,queue,priority,language,tag_1,tag_2,tag_3,tag_4,tag_5,tag_6,tag_7,tag_8
0,Unvorhergesehener Absturz der Datenanalyse-Pla...,Die Datenanalyse-Plattform brach unerwartet ab...,Ich werde Ihnen bei der Lösung des Problems he...,Incident,General Inquiry,low,de,Crash,Technical,Bug,Hardware,Resolution,Outage,Documentation,NaN
1,Customer Support Inquiry,Seeking information on digital strategies that...,We offer a variety of digital strategies and s...,Request,Customer Service,medium,en,Feedback,Sales,IT,Tech Support,NaN,NaN,NaN,NaN
2,Data Analytics for Investment,I am contacting you to request information on ...,I am here to assist you with data analytics to...,Request,Customer Service,medium,en,Technical,Product,Guidance,Documentation,Performance,Feature,NaN,NaN
3,Krankenhaus-Dienstleistung-Problem,Ein Medien-Daten-Sperrverhalten trat aufgrund ...,Zurück zur E-Mail-Beschwerde über den Sperrver...,Incident,Customer Service,high,de,Security,Breach,Login,Maintenance,Incident,Resolution,Feedback,NaN
4,Security,"Dear Customer Support, I am reaching out to in...","Dear [name], we take the security of medical d...",Request,Customer Service,medium,en,Security,Customer,Compliance,Breach,Documentation,Guidance,NaN,NaN


In [2]:
!pip install -q sentence-transformers xgboost lightgbm catboost scikit-learn imbalanced-learn

In [3]:
conteo_prioridad = df['priority'].value_counts()
print(conteo_prioridad)


priority
medium    8144
high      7801
low       4055
Name: count, dtype: int64


In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, accuracy_score, f1_score
from sklearn.linear_model import LogisticRegression
from sentence_transformers import SentenceTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from imblearn.over_sampling import SMOTE
import xgboost as xgb
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
import joblib
import warnings
warnings.filterwarnings("ignore")

Limpieza de los datos, eliminamos datos nulos

In [5]:
# df = df.dropna(subset=["subject", "body", "priority"])
# df["text"] = df["subject"] + " " + df["body"]
# le = LabelEncoder()
# y = le.fit_transform(df["priority"])


In [6]:
df = df.dropna(subset=["subject", "body", "priority"])
df["text"] = df["body"]
le = LabelEncoder()
y = le.fit_transform(df["priority"])


Generación de embeddings


In [7]:
model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
embeddings = model.encode(df["text"].tolist(), show_progress_bar=True)


Batches:   0%|          | 0/580 [00:00<?, ?it/s]

In [8]:
# model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

# embeddings = model.encode(df["text"].tolist(), show_progress_bar=True)
# print(np.array(embeddings).shape)

In [9]:
# model = SentenceTransformer("paraphrase-multilingual-mpnet-base-v2")
# embeddings = model.encode(df["text"].tolist(), show_progress_bar=True)

In [10]:
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(embeddings, y)

print("Antes del SMOTE:", np.bincount(y))
print("Después del SMOTE:", np.bincount(y_resampled))

Antes del SMOTE: [7198 3787 7552]
Después del SMOTE: [7552 7552 7552]


División del conjunto

In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X_resampled, y_resampled, test_size=0.2, stratify=y_resampled, random_state=42
)

Modelos a usar

In [ ]:
xgb_param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [4, 6, 8],
    "learning_rate": [0.01, 0.05, 0.1],
    "subsample": [0.6, 0.8, 1.0],
    "colsample_bytree": [0.6, 0.8, 1.0]
}

xgb_base = xgb.XGBClassifier(use_label_encoder=False, eval_metric="mlogloss", random_state=42)
xgb_search = RandomizedSearchCV(xgb_base, param_distributions=xgb_param_grid,
                                n_iter=20, cv=3, scoring="f1_macro", verbose=1, n_jobs=-1)
xgb_search.fit(X_train, y_train)
best_xgb = xgb_search.best_estimator_
print("Mejores hiperparámetros XGBoost:", xgb_search.best_params_)

Fitting 3 folds for each of 20 candidates, totalling 60 fits


In [ ]:
models = {
    "Random Forest": RandomForestClassifier(n_estimators=300, max_depth=20, class_weight='balanced', random_state=42),
    "MLP Classifier": MLPClassifier(hidden_layer_sizes=(256, 128), activation='relu', solver='adam', max_iter=300, random_state=42),
    "XGBoost (Optimizado)": best_xgb,
    "LightGBM": LGBMClassifier(n_estimators=300, learning_rate=0.05, max_depth=8, random_state=42),
    "CatBoost": CatBoostClassifier(iterations=300, learning_rate=0.05, depth=8, verbose=0, random_state=42),
    "Logistic Regression": LogisticRegression(max_iter=500, class_weight='balanced', solver='lbfgs')
}


Selección

In [ ]:
results = []

for name, clf in models.items():
    print(f"\n{name}\n{'-' * len(name)}")
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average="macro")
    print(classification_report(y_test, y_pred, target_names=le.classes_))

    # Confusion matrix
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le.classes_)
    disp.plot(cmap="Blues", values_format="d")
    plt.title(f"Matriz de confusión: {name}")
    plt.show()

    results.append({
        "Modelo": name,
        "Accuracy": acc,
        "F1 macro": f1
    })

In [ ]:
summary_df = pd.DataFrame(results).sort_values("F1 macro", ascending=False)
print("\n Comparativa final de modelos:\n")
print(summary_df)

Guardamos el modelo

In [ ]:
import joblib

best_model_info = max(results, key=lambda x: x["F1 macro"])
best_model_name = best_model_info["Modelo"]
print(f"Mejor modelo: {best_model_name} (F1 macro: {best_model_info['F1 macro']:.4f})")

best_model = models[best_model_name]

joblib.dump(best_model, f"Priocheck_{best_model_name.replace(' ', '_').lower()}.pkl")
print(f"Modelo guardado como: Priocheck_{best_model_name.replace(' ', '_').lower()}.pkl")
